# 02 · EDA: Capacidad de gerentes y tiempos por actividad

**Prueba analítica DICAGI 2022 · Bancolombia**

---

## 📌 Objetivo del notebook

Construir el insumo numérico más importante del modelo: **el tiempo que cada cliente le demandaría a su gerente durante el año**, comparado contra la **capacidad disponible** de los gerentes.

## 🧠 Mindset

> *"En problemas con restricciones, el cuello de botella manda. Encuéntralo antes de optimizar."*

Sin esta cuantificación no hay restricción de capacidad, y sin restricción de capacidad el problema es trivial (todos los clientes asignados, todos los gerentes saturados, métrica = 100%). **La capacidad es lo que hace este problema interesante y difícil.**

## 🎓 Las tres preguntas que respondemos

1. **¿Cuánta capacidad anual hay disponible?** (oferta agregada)
2. **¿Cuánto tiempo demanda cada cliente?** (demanda individual)
3. **¿Cuál es el ratio demanda/oferta?** Si > 1, hay clientes que **necesariamente** quedan sin asignar — el modelo elige cuáles.

## 📐 Ecuación clave que vamos a operacionalizar

Para cada cliente $c$:

$$\tau_c = \sum_{p \in P_c} \Big[ \underbrace{\mathbb{1}[\text{no tiene}]}_{\text{venta nueva}} \big(t^{venta}_p + t^{instr}_p\big) + \underbrace{\mathbb{1}[\text{tiene-no usa}]}_{\text{activación}} t^{post}_p \Big]$$

Donde:
- $P_c$ = productos con oportunidad para el cliente $c$ (de `pcac_oportunidades_comer`).
- $\mathbb{1}[\cdot]$ = indicador 0/1 (de `pcac_mac_gpi_tenencia_prod`).
- $t^{etapa}_p$ = mediana de minutos por actividad (de `pcac_encuesta`).

Y para cada ejecutivo $e$ (la unidad atómica del modelo):

$$t_e = \sum_{c \in C_e} \tau_c$$

## 🎓 Las tres capas de explicación

Cada sección lleva los iconos:
- 👥 **Versión simple** (público no técnico)
- 🔬 **Versión técnica** (analista / data scientist)
- ⚛️ **Análogo físico** (intuición de física estadística)

## Tablas usadas

| Tabla | Filas | Rol |
|---|---|---|
| `pcac_capacidad_gerentes.csv` | 50 | Tiempo anual disponible $T_g$ por gerente |
| `pcac_encuesta.csv` | ~1,805 | Minutos por (gerente, producto, etapa) |
| `pcac_oportunidades_comer.csv` | ~247,864 | Productos potenciales por cliente |
| `pcac_mac_gpi_tenencia_prod.csv` | ~66,803 | Si el cliente ya tiene/usa cada producto |

---
## 0. Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from modelo_capacidad.data.loader import (
    load_capacidad, load_encuesta, load_oportunidades, load_tenencia, load_clientes,
)
from modelo_capacidad.viz import apply_theme, BANCOLOMBIA_COLORS, BANCOLOMBIA_SEQUENTIAL

apply_theme()
print('Entorno listo.')

In [ ]:
capacidad = load_capacidad()
encuesta = load_encuesta()
oport = load_oportunidades()
tenencia = load_tenencia()
clientes = load_clientes()

for n, df in [('capacidad', capacidad), ('encuesta', encuesta),
              ('oport', oport), ('tenencia', tenencia), ('clientes', clientes)]:
    print(f'{n:>12}: {df.shape[0]:>7,} × {df.shape[1]:>2}')
capacidad.head()

---
## 1. Capacidad disponible — ¿con cuánto tiempo contamos?

### 🎯 ¿Qué hacemos?
Inspeccionamos la tabla `pcac_capacidad_gerentes`, que define para cada gerente:
- `sistematica_anual`: el tiempo total que el gerente dedica al año (típicamente 85,050 min ≈ 1,418 horas).
- `tiempo_instrum_resta`: tiempo descontado por instrumentación de productos pre-existentes.
- `tiempo_restante = sistematica_anual − tiempo_instrum_resta`: el tiempo **realmente disponible** para asignar nuevos clientes.

### 🧠 Mindset

> *"Una capacidad expresada como un solo número esconde supuestos. Entiende esos supuestos antes de tratarla como ley."*

El número 85,050 minutos ≈ **1,418 horas/año** ≈ **177 días laborables × 8h** parece casi todo el año laboral, lo cual ya levanta una pregunta: ¿es "tiempo neto comercial" o incluye reuniones internas, capacitaciones, vacaciones? Para este modelo lo asumimos como tiempo realmente disponible para clientes (es el dato que nos dan).

### 👥 Versión simple
Cada gerente tiene una "chequera de minutos" para gastar en clientes durante el año. La chequera trae 75,930 minutos disponibles (después de descontar lo que ya está comprometido). Cuando le asignamos un cliente, restamos sus minutos. Cuando llega a cero, **no acepta más**.

### 🔬 Versión técnica
$T_g$ es la **capacidad efectiva** del gerente $g$. Aparece como restricción de mochila ponderada en el MILP:
$$\sum_{e \in E_g} t_e \cdot x_{e,g} \le T_g$$

Una alternativa que **rechazamos**: usar `sistematica_anual` directamente (85,050) ignorando el descuento. Esto sobrestimaría capacidad y haría el modelo "infactible silenciosamente" en producción (gerentes saturados que en la práctica no pueden con la carga).

### ⚛️ Análogo físico
$T_g$ es la **"profundidad del pozo de potencial"** del gerente $g$: hasta cuánta "masa" (tiempo de clientes) puede atraer antes de que el sistema rechace más. En spin-glass-with-capacity, esto se modela con una penalización cuadrática $\lambda_g (\sum_e t_e x_{e,g} - T_g)_+^2$ donde $(\cdot)_+$ es la rampa positiva.

In [ ]:
# Coherencia interna: tiempo_restante debería = sistematica_anual − tiempo_instrum_resta
cap = capacidad.copy()
cap['restante_calculado'] = cap['sistematica_anual'] - cap['tiempo_instrum_resta'].fillna(0)
cap['delta'] = cap['tiempo_restante'] - cap['restante_calculado']
print('Coherencia tiempo_restante = sistematica − instrum:')
print(f'  delta máximo: {cap["delta"].abs().max()}')
print(f'  filas inconsistentes: {(cap["delta"].abs() > 1).sum()}')
print()
print('Distribución de capacidad:')
capacidad[['sistematica_anual', 'tiempo_instrum_resta', 'tiempo_restante']].describe().round(0)

In [ ]:
# Capacidad agregada por región (oferta total disponible)
cap_region = (capacidad
    .groupby('cod_region_gte_inv')
    .agg(gerentes=('cod_gte_inv', 'nunique'),
         capacidad_total=('tiempo_restante', 'sum'),
         capacidad_media=('tiempo_restante', 'mean'))
    .reset_index())
cap_region['capacidad_horas'] = (cap_region['capacidad_total'] / 60).round(0)

fig = px.bar(
    cap_region.sort_values('capacidad_total', ascending=False),
    x='cod_region_gte_inv', y='capacidad_total', text='gerentes',
    title='Oferta de tiempo anual por región (minutos)',
    color_discrete_sequence=[BANCOLOMBIA_COLORS['amarillo']],
)
fig.update_traces(texttemplate='%{text} gerentes', textposition='outside')
fig.update_layout(xaxis_title='Código región', yaxis_title='Minutos disponibles / año')
fig.show()
cap_region

### 📏 Magnitudes de referencia

- 1 gerente ≈ 75,930 min/año ≈ **1,265 horas** ≈ **158 jornadas de 8h**.
- 50 gerentes × 75,930 = **3,796,500 min/año** = total nacional de oferta.
- Si la demanda total resultara similar a este número, el problema está "en el filo". Si la demanda es 2× la oferta, **el 50% de los clientes queda fuera** sí o sí (independiente del modelo).

**Si una región tiene 2 gerentes y otra tiene 18**, la capacidad regional varía en 9× — y la dificultad relativa también.

---
## 2. Tiempos por actividad y producto — la "física" microscópica

### 🎯 ¿Qué hacemos?
Calculamos la **mediana** de minutos por (producto, etapa) usando `pcac_encuesta`.

### 🧠 ¿Por qué mediana y no media?
La encuesta tiene respuestas auto-reportadas de gerentes. Los outliers (alguien que reporta 480 min para vender una cuenta de ahorros) inflan la media. **La mediana es robusta** a estos errores y mejor representa el caso típico.

**Alternativas consideradas:**
- *Media truncada* (eliminar top y bottom 5%): da resultados muy parecidos a la mediana en datos sesgados; preferimos la mediana por simplicidad.
- *Mediana por región*: usar la mediana del tiempo según la región del gerente. Más fina, pero la encuesta tiene pocos datos por región — preferimos mediana global por producto/etapa.
- *Tiempo del gerente específico que va a recibir la asignación*: lo más fiel pero genera dependencia circular en el modelo (qué gerente recibe depende del tiempo, que depende del gerente). **Antipatrón**.

### 👥 Versión simple
Es como tener una receta de cocina: cada producto del banco requiere ciertos pasos (vender, instrumentar, hacer post-venta) y cada paso toma un tiempo "típico". En vez de creerle a cualquier cocinero (algunos exageran), tomamos el **tiempo de la mitad de los cocineros** — el del medio, ni los rapidísimos ni los lentísimos.

### 🔬 Versión técnica
Estamos construyendo un **lookup table** $t^{etapa}_p$ con dos llaves (producto, etapa). En SQL sería:
```sql
SELECT producto, etapa_del_producto,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY total_promedio_tiempo_min_x_actividad) AS minutos_mediana
FROM pcac_encuesta
GROUP BY producto, etapa_del_producto;
```

### ⚛️ Análogo físico
$t^{etapa}_p$ son los **"costos energéticos elementales"** del sistema: cada interacción microscópica (vender producto X, instrumentar producto Y) tiene un costo energético constante. La energía total del sistema (tiempo total demandado) es la suma sobre todas las interacciones — análoga a una **suma sobre Hamiltoniano de N-cuerpos sin acoplamiento entre partículas**.

In [ ]:
tiempos_por_etapa = (encuesta
    .dropna(subset=['producto', 'etapa_del_producto', 'total_promedio_tiempo_min_x_actividad'])
    .groupby(['producto', 'etapa_del_producto'])['total_promedio_tiempo_min_x_actividad']
    .agg(mediana='median', media='mean', n='count')
    .round(1)
    .reset_index())

print(f'Combinaciones (producto, etapa) únicas: {len(tiempos_por_etapa)}')
print(f'Etapas presentes: {tiempos_por_etapa["etapa_del_producto"].unique().tolist()}')
tiempos_por_etapa.head(15)

In [ ]:
# Top 25 actividades más costosas (mediana)
fig = px.bar(
    tiempos_por_etapa.sort_values('mediana', ascending=True).tail(25),
    x='mediana', y='producto', color='etapa_del_producto',
    orientation='h',
    title='Top 25 actividades por mediana de minutos',
    color_discrete_sequence=[BANCOLOMBIA_COLORS['amarillo'], BANCOLOMBIA_COLORS['negro'], BANCOLOMBIA_COLORS['gris']],
    hover_data=['n', 'media'],
)
fig.update_layout(yaxis_title=None, xaxis_title='Minutos / actividad', legend_title='Etapa', height=600)
fig.show()

### 📏 Lectura

- **Productos top en tiempo**: típicamente activos complejos (renta variable, vinculaciones offshore, productos estructurados). Pueden costar 60-120 min por actividad.
- **Productos baratos**: cuentas de ahorro, tarjetas, productos transversales. 5-30 min.
- **Diferencia entre mediana y media**: si la media es >1.5× mediana, hay outliers fuertes (recordar: descartamos esos al usar mediana).

**Implicación de modelo**: clientes con muchas oportunidades en productos caros van a saturar gerentes rápido. Si un cliente A tiene 5 oportunidades en productos de 90 min cada una × 3 etapas = 1,350 min — eso es **1.8% de la capacidad anual** de un gerente entero.

---
## 3. Distribución de oportunidades por cliente

### 🎯 ¿Qué hacemos?
Contamos cuántas oportunidades de venta tiene cada cliente.

### 🧠 ¿Por qué importa?
Un cliente con 0 oportunidades no genera demanda → asignarlo es "barato" pero también no aporta valor de venta. Un cliente con 20 oportunidades es valioso pero **demanda mucho tiempo**.

Estos dos extremos hacen que el ratio **valor/tiempo** sea variable, lo que justifica un modelo (no se puede priorizar solo por categoría).

### 👥 Versión simple
Algunos clientes son como un buffet (muchos productos por venderles), otros como un café rápido. Ambos cuentan en la métrica si los asignamos, pero el buffet **gasta mucho tiempo del gerente**.

### 🔬 Versión técnica
Distribución empírica del número de oportunidades por cliente. Esperamos cola larga (Pareto-like). Reportamos cuantiles para parametrizar el modelo.

### ⚛️ Análogo físico
El número de oportunidades es proporcional al **acoplamiento** del cliente con el sistema productivo. Clientes muy acoplados aportan mucha energía pero también demandan mucha — la **interacción** es fuerte en ambos sentidos.

In [ ]:
oport_cli = (oport.groupby('num_doc_cli').size().reset_index(name='n_oportunidades'))

fig = px.histogram(
    oport_cli, x='n_oportunidades', nbins=40,
    title='# oportunidades comerciales por cliente',
    color_discrete_sequence=[BANCOLOMBIA_COLORS['amarillo']],
)
fig.add_vline(x=oport_cli['n_oportunidades'].median(), line_dash='dash',
              line_color=BANCOLOMBIA_COLORS['negro'],
              annotation_text=f'mediana={int(oport_cli["n_oportunidades"].median())}')
fig.update_layout(xaxis_title='# oportunidades por cliente', yaxis_title='# clientes', bargap=0.05)
fig.show()

print('Cuantiles del # oportunidades por cliente:')
oport_cli['n_oportunidades'].describe(percentiles=[0.5, 0.75, 0.9, 0.99]).round(1)

### 📏 Diagnóstico

Si la distribución es de cola larga (P99 >> P50), los clientes "top" son **outliers en demanda**. Vale la pena identificarlos: son los más valiosos (más venta potencial) pero también los que pueden saturar gerentes.

---
## 4. Demanda total estimada vs capacidad — el ratio crítico

### 🎯 ¿Qué hacemos?
Calculamos una **estimación gruesa** de la demanda total (sumando aproximadamente $\tau_c$ sobre todos los clientes) y la comparamos con la oferta nacional de tiempo.

### 🧠 ¿Por qué este es el momento más importante del notebook?
Este ratio define **si el problema tiene solución completa o no**:

| Ratio demanda/oferta | Diagnóstico | Implicación de modelo |
|---|---|---|
| < 0.7 | Holgura | El modelo asigna casi todos los clientes, métrica > 0.9 fácil. |
| 0.8 – 1.2 | En el filo | El modelo es no trivial: hay que elegir bien quién entra. |
| > 1.5 | Saturación | El cuello de botella domina. La métrica está acotada por (oferta/demanda) × pct(A+B). |

### 👥 Versión simple
Si la pizzería tiene cocina para 100 pizzas al día y le llegan 80 pedidos, está bien. Si le llegan 300, **forzosamente rechaza 200** sin importar qué tan buenos sean los pizzeros. La métrica final está acotada por la realidad física, no por el algoritmo.

### 🔬 Versión técnica
Demanda agregada estimada (estimación de baja-fidelidad antes de implementar `tiempo_demanda.py`):
$$\hat{D} = N_{\text{clientes}} \cdot \bar{n}_{\text{oportunidades por cliente}} \cdot \bar{t}_{\text{venta}}$$

**Esto es una cota superior porque ignora**: que algunos clientes ya tienen el producto (no se contabiliza tiempo de venta), y la frecuencia anual de algunas actividades.

### ⚛️ Análogo físico
Es la **densidad relativa** del sistema. En física estadística:
- Densidad baja → fase gas (movimiento libre, problema fácil).
- Densidad media → fase líquida (interacciones sí pero hay holgura).
- Densidad alta → fase sólida cristalina (todo apretado, restricciones dominan).
- Densidad ultra-alta → frustración (no hay configuración que satisfaga todo).

El ratio demanda/oferta es el equivalente al **factor de empaquetamiento** del sistema.

In [ ]:
# Estimación gruesa: tiempo medio de venta por oportunidad × total de oportunidades
tiempo_venta_mediano = (encuesta.query("etapa_del_producto == 'Venta'")
                        ['total_promedio_tiempo_min_x_actividad'].median())
n_oportunidades_total = len(oport)

demanda_aprox = tiempo_venta_mediano * n_oportunidades_total
oferta_total = capacidad['tiempo_restante'].sum()

print(f'Tiempo mediano por venta:    {tiempo_venta_mediano:>12,.0f} min')
print(f'# oportunidades totales:     {n_oportunidades_total:>12,}')
print(f'Demanda aprox (solo ventas): {demanda_aprox:>12,.0f} min')
print(f'Oferta total (capacidad):    {oferta_total:>12,.0f} min')
print(f'Ratio demanda/oferta:        {demanda_aprox / oferta_total:>12,.2f}')
print()
print('⚠️  Esta estimación NO incluye tiempo de instrumentación ni post-venta.')
print('    La demanda real probablemente es 1.5-2× esta estimación.')

In [ ]:
# Visualización demanda vs oferta por región
# (estimación: demanda regional ≈ pct_oportunidades_region × demanda_total)
oport_clientes = oport.merge(
    clientes[['num_doc_cli', 'cod_region_gte_inv']].drop_duplicates('num_doc_cli'),
    on='num_doc_cli', how='left',
)
oport_region = oport_clientes.groupby('cod_region_gte_inv').size().reset_index(name='n_oport')
oport_region['demanda_aprox'] = oport_region['n_oport'] * tiempo_venta_mediano

comp = oport_region.merge(
    cap_region[['cod_region_gte_inv', 'capacidad_total']],
    on='cod_region_gte_inv', how='outer',
).fillna(0)
comp['ratio'] = (comp['demanda_aprox'] / comp['capacidad_total']).round(2)

fig = go.Figure()
fig.add_bar(name='Demanda aprox.', x=comp['cod_region_gte_inv'], y=comp['demanda_aprox'],
            marker_color=BANCOLOMBIA_COLORS['negro'])
fig.add_bar(name='Oferta (capacidad)', x=comp['cod_region_gte_inv'], y=comp['capacidad_total'],
            marker_color=BANCOLOMBIA_COLORS['amarillo'])
fig.update_layout(
    barmode='group',
    title='Demanda estimada vs Oferta de tiempo por región',
    xaxis_title='Código región', yaxis_title='Minutos / año',
    legend_title=None,
)
fig.show()
comp.sort_values('ratio', ascending=False)

### 📏 Decisión de modelado por escenario

**Escenario A · Holgura (ratio < 0.7 en todas las regiones)**
→ Greedy simple es suficiente. MILP da poca mejora marginal.

**Escenario B · Tensión moderada (0.7 ≤ ratio < 1.5)** *(probable)*
→ MILP exacto vale la pena. Greedy puede dejar 5-10% de score sobre la mesa.

**Escenario C · Saturación severa (ratio ≥ 1.5)**
→ La métrica está acotada estructuralmente. Reportar ese techo en el documento. El modelo solo elige dentro del techo. Aquí los pesos $w_A, w_B$ tienen máximo impacto.

---
## 5. Tenencia y oportunidades — ¿se ofrecen productos que el cliente ya tiene?

### 🎯 ¿Qué hacemos?
Cruzamos `pcac_oportunidades_comer` con `pcac_mac_gpi_tenencia_prod` para detectar el % de oportunidades **redundantes** (productos ya tenidos por el cliente).

### 🧠 ¿Por qué importa para el modelo?
La fórmula $\tau_c$ tiene tres ramas:
1. Cliente NO tiene el producto → suma `t_venta + t_instrumentación` (caso típico, alto costo).
2. Cliente SÍ tiene pero NO usa → suma solo `t_post_venta` (activación, costo medio).
3. Cliente SÍ tiene Y SÍ usa → costo cero (no hay nada que ofrecer).

Si la mayoría de oportunidades son del tipo 1 (clientes que no tienen), el tiempo demandado es alto. Si muchas son tipo 3, la demanda real es **mucho menor que la estimación gruesa**.

### 👥 Versión simple
Si una zapatería te llama para venderte unos zapatos que tú **ya compraste el mes pasado**, perdió tiempo. Si en cambio te llama porque te compraste pero no estrenas (oportunidad de "activación"), gasta menos tiempo. Lo más caro es venderte algo nuevo desde cero.

### 🔬 Versión técnica
`merge(oport, tenencia, on=['num_doc_cli', 'cod_producto'], how='left')` y luego clasificar por las flags `tenencia` y `usa_producto`.

### ⚛️ Análogo físico
Es la **función de partición efectiva**: no todas las interacciones contribuyen igual a la energía. Algunas son "silenciosas" (cliente ya tiene y usa) y otras tienen costo completo (cliente nuevo). Es análogo a tener **estados degenerados** en mecánica cuántica — no aportan energía pero existen.

In [ ]:
# Cruce oportunidades x tenencia (por cliente y producto)
merged = oport.merge(
    tenencia[['num_doc_cli', 'cod_prod', 'tenencia', 'usa_producto']]
        .rename(columns={'cod_prod': 'cod_producto'}),
    on=['num_doc_cli', 'cod_producto'], how='left',
)

def clasificar(row):
    if pd.isna(row['tenencia']) or row['tenencia'] == 0:
        return '1. No tiene → venta nueva'
    if row.get('usa_producto', 0) in (0, np.nan) or pd.isna(row.get('usa_producto')):
        return '2. Tiene pero no usa → activación'
    return '3. Tiene y usa → sin tiempo'

merged['caso_tau'] = merged.apply(clasificar, axis=1)
casos = merged['caso_tau'].value_counts(normalize=True).mul(100).round(1).reset_index()
casos.columns = ['caso', 'pct']

fig = px.bar(
    casos, x='caso', y='pct', text='pct',
    title='Distribución de oportunidades según tenencia',
    color_discrete_sequence=[BANCOLOMBIA_COLORS['amarillo']],
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(xaxis_title=None, yaxis_title='% de oportunidades')
fig.show()
casos

### 📏 Lectura

- Si **>70% son caso 1** (venta nueva): la fórmula completa de $\tau_c$ es necesaria; la demanda es alta.
- Si hay un **% no despreciable de caso 3** (ya tiene y usa): cuidado, hay que **excluirlas** de la suma — sino sobre-estimamos la demanda.
- El caso 2 puede ser pequeño en volumen pero alto en valor (cliente activado = ingreso extra).

---
## 6. Implicaciones para el modelo

**Resumen de outputs de este notebook que el modelo va a consumir:**

| Output | Dónde se calcula formalmente | Variable en el modelo |
|---|---|---|
| Capacidad por gerente $T_g$ | `pcac_capacidad_gerentes.tiempo_restante` directo | RHS de la restricción de capacidad |
| Tiempo por (producto, etapa) | `tiempos_por_etapa` (mediana) | Lookup para construir $\tau_c$ |
| Demanda por cliente $\tau_c$ | `src/modelo_capacidad/features/tiempo_demanda.py` (a implementar) | Componente de $t_e$ |
| Demanda por ejecutivo $t_e$ | Suma sobre $C_e$ | Coeficiente de la variable $x_{e,g}$ |
| Ratio demanda/oferta global | Diagnóstico aquí | Define si problema es saturado |

### Próximo paso

Implementar `features/tiempo_demanda.py` con función:
```python
def calcular_tau_cliente(oport: pd.DataFrame, tenencia: pd.DataFrame, tiempos: pd.DataFrame) -> pd.DataFrame:
    """Devuelve DataFrame[num_doc_cli, tau] con el tiempo total anual demandado por cada cliente."""
```
y luego `t_ejecutivo = clientes.merge(tau).groupby('cod_ejec_bco')['tau'].sum()`.

Esto cierra el insumo del modelo (notebook 03) y del Hamiltoniano (notebook 04).